# Testing file 
### where we evaluate Zhang's models using the test set

## Preliminaries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.data import Dataset


from util.load_data import load_data
from util.evaluation import *
from models.zhang.models import FairLogisticRegression
from models.zhang.learning import train_loop as zhang_train

/Users/lffpl/Projects/falsb/env/falsb/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
batch_size = 64
epochs = 100
lr = 0.001

In [3]:
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'balanced-arrhythmia'

In [5]:
x, y, a = load_data(data_name)
raw_data = (x, y, a)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
adim = a.shape[1]
zdim = 8

In [7]:
print(xdim, ydim, adim, zdim)

278 1 1 8


In [8]:
print(len(x))

474


## Result file

In [9]:
header = "model_name", "cv_seed", "clas_acc", "dp", "deqodds", "deqopp", "trade_dp", "trade_deqodds", "trade_deqopp", "TN_a0", "FP_a0", "FN_a0", "TP_a0", "TN_a1", "FP_a1", "FN_a1", "TP_a1"
results = []

## Testing loop
#### Each model is evalueted 5 times
#### In the end of each iteration we save the result

### Zhang for DP

In [10]:
fairdef = 'DemPar'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)

    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4DP', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc


2026-01-04 11:26:40.993579: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-01-04 11:26:41.131083: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 1 | 0.7281466722488403 | 0.7525843381881714 | 0.528125 | 0.540625
> 2 | 0.6948592662811279 | 0.7517109513282776 | 0.528125 | 0.540625
> 3 | 0.6877017617225647 | 0.7509129643440247 | 0.55 | 0.540625
> 4 | 0.6825787425041199 | 0.7500299215316772 | 0.5625 | 0.540625


2026-01-04 11:26:41.369284: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 5 | 0.6775645017623901 | 0.7491779327392578 | 0.596875 | 0.540625
> 6 | 0.6726468801498413 | 0.7483381032943726 | 0.63125 | 0.540625
> 7 | 0.6678421497344971 | 0.7475162744522095 | 0.640625 | 0.540625
> 8 | 0.6631824374198914 | 0.7467186450958252 | 0.653125 | 0.540625


...

> 65 | 0.5147114992141724 | 0.7171962857246399 | 0.771875 | 0.540625
> 66 | 0.5131694078445435 | 0.7167834639549255 | 0.771875 | 0.540625
> 67 | 0.5116472840309143 | 0.7163577079772949 | 0.771875 | 0.540625
> 68 | 0.510199248790741 | 0.715947687625885 | 0.771875 | 0.540625
> 69 | 0.5087656378746033 | 0.7155332565307617 | 0.775 | 0.540625
> 70 | 0.5073628425598145 | 0.7151332497596741 | 0.778125 | 0.540625
> 71 | 0.5059821605682373 | 0.7147247791290283 | 0.775 | 0.540625
> 72 | 0.5046184062957764 | 0.714328944683075 | 0.778125 | 0.540625
> 73 | 0.5033264756202698 | 0.7139348983764648 | 0.778125 | 0.540625
> 74 | 0.502062976360321 | 0.7135355472564697 | 0.78125 | 0.540625
> 75 | 0.5008156299591064 | 0.7131357192993164 | 0.78125 | 0.540625
> 76 | 0.4995824098587036 | 0.7127182483673096 | 0.784375 | 0.540625
> 77 | 0.4983752965927124 | 0.7122902274131775 | 0.784375 | 0.540625
> 78 | 0.49720442295074463 | 0.7118692994117737 | 0.78125 | 0.540625
> 79 | 0.4960528314113617 | 0.7114337682723999

2026-01-04 11:26:55.928617: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 27 | 0.6362370252609253 | 0.7419326305389404 | 0.765625 | 0.528125
> 28 | 0.6350156664848328 | 0.7414240837097168 | 0.7625 | 0.528125
> 29 | 0.6326404809951782 | 0.7405829429626465 | 0.76875 | 0.528125
> 30 | 0.6314889788627625 | 0.740157425403595 | 0.76875 | 0.528125
> 31 | 0.6291171908378601 | 0.7393428087234497 | 0.775 | 0.528125
> 32 | 0.6279487609863281 | 0.738864541053772 | 0.771875 | 0.528125
> 33 | 0.6256507635116577 | 0.7380658388137817 | 0.771875 | 0.528125
> 34 | 0.6245088577270508 | 0.737617552280426 | 0.771875 | 0.528125
> 35 | 0.622482180595398 | 0.7369003295898438 | 0.775 | 0.528125
> 36 | 0.6211421489715576 | 0.7363771200180054 | 0.771875 | 0.528125
> 37 | 0.6194278597831726 | 0.7357426285743713 | 0.775 | 0.528125
> 38 | 0.6179467439651489 | 0.7351856231689453 | 0.775 | 0.528125
> 39 | 0.6163073778152466 | 0.7345936894416809 | 0.775 | 0.528125
> 40 | 0.6147899031639099 | 0.7341004014015198 | 0.775 | 0.528125
> 41 | 0.6132147312164307 | 0.7335770130157471 | 0.775 | 0.5

2026-01-04 11:27:11.588608: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 54 | 0.5541066527366638 | 0.5992686748504639 | 0.75625 | 0.571875
> 55 | 0.5528515577316284 | 0.5988814830780029 | 0.753125 | 0.571875
> 56 | 0.551611065864563 | 0.5984899401664734 | 0.753125 | 0.571875
> 57 | 0.5504030585289001 | 0.5981099009513855 | 0.753125 | 0.571875
> 58 | 0.549231767654419 | 0.5977195501327515 | 0.759375 | 0.571875
> 59 | 0.5480484366416931 | 0.5973483324050903 | 0.753125 | 0.571875
> 60 | 0.5469486117362976 | 0.5969606637954712 | 0.75625 | 0.571875
> 61 | 0.5458309054374695 | 0.5965955257415771 | 0.753125 | 0.571875
> 62 | 0.5448108911514282 | 0.5962084531784058 | 0.753125 | 0.571875
> 63 | 0.5437690019607544 | 0.5958422422409058 | 0.753125 | 0.571875
> 64 | 0.5427644848823547 | 0.5954630970954895 | 0.759375 | 0.571875
> 65 | 0.5417261123657227 | 0.5951099395751953 | 0.759375 | 0.571875
> 66 | 0.5408608913421631 | 0.594714879989624 | 0.759375 | 0.571875
> 67 | 0.5398310422897339 | 0.5943752527236938 | 0.7625 | 0.571875
> 68 | 0.539010763168335 | 0.593981623649

### Zhang for Eq Odds

In [11]:
fairdef = 'EqOdds'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.7281466722488403 | 0.7519299983978271 | 0.528125 | 0.540625
> 2 | 0.6948639750480652 | 0.750699520111084 | 0.528125 | 0.540625
> 3 | 0.6876965761184692 | 0.7495851516723633 | 0.55 | 0.540625
> 4 | 0.6826187372207642 | 0.7482991218566895 | 0.559375 | 0.540625
> 5 | 0.6776115298271179 | 0.7470695972442627 | 0.590625 | 0.540625
> 6 | 0.6727321147918701 | 0.745876669883728 | 0.625 | 0.540625
> 7 | 0.6679269075393677 | 0.7447688579559326 | 0.64375 | 0.540625


2026-01-04 11:27:43.070460: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 8 | 0.6632416248321533 | 0.7436962723731995 | 0.653125 | 0.540625
> 9 | 0.6587185859680176 | 0.7426633834838867 | 0.665625 | 0.540625
> 10 | 0.6542770266532898 | 0.7416653633117676 | 0.678125 | 0.540625
> 11 | 0.6499955058097839 | 0.7406902313232422 | 0.6875 | 0.540625
> 12 | 0.6457909345626831 | 0.739764928817749 | 0.7 | 0.540625
> 13 | 0.6416628956794739 | 0.7388714551925659 | 0.715625 | 0.540625
> 14 | 0.6376513242721558 | 0.7380076050758362 | 0.7125 | 0.540625
> 15 | 0.6336992979049683 | 0.7371922731399536 | 0.721875 | 0.540625
> 16 | 0.6297950744628906 | 0.7363847494125366 | 0.715625 | 0.540625
> 17 | 0.6259744763374329 | 0.7356250286102295 | 0.71875 | 0.540625
> 18 | 0.6222426891326904 | 0.7349210977554321 | 0.725 | 0.540625
> 19 | 0.6185845136642456 | 0.7342403531074524 | 0.728125 | 0.540625
> 20 | 0.614979088306427 | 0.7335609793663025 | 0.725 | 0.540625
> 21 | 0.6114642024040222 | 0.7329398989677429 | 0.725 | 0.540625
> 22 | 0.6080731749534607 | 0.7323025465011597 | 0.725 | 

### Zhang for Eq Opp

In [12]:
fairdef = 'EqOpp'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOpp', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.7280764579772949 | 0.34793275594711304 | 0.528125 | 0.540625
> 2 | 0.6946718692779541 | 0.3474102020263672 | 0.528125 | 0.540625
> 3 | 0.6874301433563232 | 0.346915066242218 | 0.546875 | 0.540625
> 4 | 0.6822346448898315 | 0.3463646471500397 | 0.565625 | 0.540625
> 5 | 0.6771692633628845 | 0.34582918882369995 | 0.590625 | 0.540625
> 6 | 0.6721426248550415 | 0.3452969789505005 | 0.625 | 0.540625
> 7 | 0.6672293543815613 | 0.34485989809036255 | 0.65 | 0.540625
> 8 | 0.6624364852905273 | 0.34442877769470215 | 0.66875 | 0.540625
> 9 | 0.6577255129814148 | 0.34400272369384766 | 0.66875 | 0.540625
> 10 | 0.6531802415847778 | 0.34358179569244385 | 0.68125 | 0.540625
> 11 | 0.6487649083137512 | 0.34316468238830566 | 0.696875 | 0.540625
> 12 | 0.64445960521698 | 0.34275129437446594 | 0.715625 | 0.540625
> 13 | 0.6402344703674316 | 0.34234896302223206 | 0.715625 | 0.540625
> 14 | 0.6361075639724731 | 0.34194374084472656 | 0.71875 | 0.

2026-01-04 11:28:44.571097: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 15 | 0.6320723295211792 | 0.3415408134460449 | 0.725 | 0.540625
> 16 | 0.6280537843704224 | 0.3411390483379364 | 0.728125 | 0.540625
> 17 | 0.6241418123245239 | 0.34075403213500977 | 0.725 | 0.540625
> 18 | 0.6202942132949829 | 0.3404330015182495 | 0.7375 | 0.540625
> 19 | 0.6165239810943604 | 0.3401143252849579 | 0.7375 | 0.540625
> 20 | 0.6128305196762085 | 0.33979886770248413 | 0.74375 | 0.540625
> 21 | 0.609224796295166 | 0.3394847512245178 | 0.7375 | 0.540625
> 22 | 0.6056900024414062 | 0.33916887640953064 | 0.746875 | 0.540625
> 23 | 0.6022849678993225 | 0.33885177969932556 | 0.746875 | 0.540625
> 24 | 0.5989783406257629 | 0.33853399753570557 | 0.746875 | 0.540625
> 25 | 0.5957152843475342 | 0.3382231891155243 | 0.746875 | 0.540625
> 26 | 0.5925320386886597 | 0.337898850440979 | 0.75 | 0.540625
> 27 | 0.589449405670166 | 0.3375753164291382 | 0.753125 | 0.540625
> 28 | 0.5864307880401611 | 0.3372499942779541 | 0.753125 | 0.540625
> 29 | 0.5834777355194092 | 0.3369308412075043 | 

## Saving into DF then CSV

In [13]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,dp,deqodds,deqopp,trade_dp,trade_deqodds,trade_deqopp,TN_a0,FP_a0,FN_a0,TP_a0,TN_a1,FP_a1,FN_a1,TP_a1
0,Zhang4DP,13,0.710938,0.730533,0.849746,0.960205,0.720602,0.774169,0.816981,13.0,15.0,7.0,34.0,29.0,11.0,4.0,15.0
1,Zhang4DP,29,0.656250,0.599694,0.673834,0.680000,0.626699,0.664926,0.667914,10.0,17.0,9.0,41.0,26.0,11.0,7.0,7.0
2,Zhang4DP,42,0.710938,0.824510,0.937410,0.918660,0.763523,0.808616,0.801560,17.0,10.0,6.0,27.0,33.0,16.0,5.0,14.0
3,Zhang4DP,55,0.773438,0.608504,0.738337,0.806818,0.681129,0.755480,0.789775,13.0,9.0,8.0,36.0,35.0,3.0,9.0,15.0
4,Zhang4DP,73,0.750000,0.703125,0.726695,0.763636,0.725806,0.738164,0.756757,15.0,16.0,1.0,32.0,27.0,7.0,8.0,22.0
5,Zhang4EqOdds,13,0.710938,0.730533,0.849746,0.960205,0.720602,0.774169,0.816981,13.0,15.0,7.0,34.0,29.0,11.0,4.0,15.0
6,Zhang4EqOdds,29,0.656250,0.599694,0.673834,0.680000,0.626699,0.664926,0.667914,10.0,17.0,9.0,41.0,26.0,11.0,7.0,7.0
7,Zhang4EqOdds,42,0.710938,0.824510,0.937410,0.918660,0.763523,0.808616,0.801560,17.0,10.0,6.0,27.0,33.0,16.0,5.0,14.0
8,Zhang4EqOdds,55,0.773438,0.608504,0.738337,0.806818,0.681129,0.755480,0.789775,13.0,9.0,8.0,36.0,35.0,3.0,9.0,15.0
9,Zhang4EqOdds,73,0.757812,0.718750,0.743361,0.796970,0.737765,0.750517,0.776898,15.0,16.0,1.0,32.0,27.0,7.0,7.0,23.0


In [14]:
result_df.to_csv(f'{data_name}-result/zhang-{epochs}.csv')